In [17]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import json

In [18]:
### Eval the results

In [19]:
#Load 4 sets of results in
#Load in with pandas
anno1 = 'combined'
anno2 = 'aime'
anno3 = 'ceci'

# suffix2 = '_enforce'
angle1 = 'avg'
angle2 = 'pizlo'

#bounds or not
bounds1 = 'bounds'
bounds2 = 'nobounds'

#fixtire or not
fixtire1 = '_fixtirefails'
fixtire2 = ''


fname_1 = f'hw7_results_{anno1}_{angle1}_{bounds1}{fixtire2}.txt'
fname_2 = f'hw7_results_{anno1}_{angle1}_{bounds1}{fixtire2}.txt'
print(fname_1,fname_2)

use_four = False
if use_four:
    fname_3 = f'hw7_results_{anno1}_{angle1}_{bounds1}{fixtire2}.txt'
    fname_4 = f'hw7_results_{anno1}_{angle1}_{bounds1}{fixtire2}.txt'
    print(fname_3,fname_4)

hw7_results_combined_avg_bounds.txt hw7_results_combined_avg_bounds.txt


In [20]:
path = Path('compare_data/results_paper_hw7_july_3')

In [21]:
df1 = pd.read_csv(path/fname_1, delim_whitespace=True, header=None)
df2 = pd.read_csv(path/fname_2, delim_whitespace=True, header=None)
if use_four:
    df3 = pd.read_csv(path/fname_3, delim_whitespace=True, header=None)
    df4 = pd.read_csv(path/fname_4, delim_whitespace=True, header=None)
#Name the columns
col_names = ["camera","target", "annotated_car_id",
       "num_sym_pairs","bbox_2D_height",
             "reproj_error","gt_heading_angle",
             "pred_heading_angle","angle_difference",
             "dist_base_gt_bbox","dist_base_pred_bbox",
             "dist_base_bbox_diff","dist_nearest_corner_gt_bbox",
             "dist_nearest_corner_pred_bbox","dist_nearest_corner_diff",
             "iou","iou_bev","mounting_height","ds",
            "PRED_OL","PRED_OW","PRED_OH","PRED_WB","LD_OL","LD_OW","LD_OH","PRED_WWOM","tire_both_sides","has_mirrors","dist_to_move",
            "LD_OW_NON","LD_OH_NON","LD_OL_NON","LENGTH_BY_GAUSSIAN","NUM_TIRES"]
assert len(col_names) == df1.shape[-1]
col_names = ["camera", "target","annotated_car_id"] + [i for i in col_names if (i != "target" and i != "annotated_car_id") and i!= "camera"]
df1.columns = col_names
df2.columns = col_names
if use_four: 
    df3.columns = col_names
    df4.columns = col_names

### Excluding cameras
df1 = df1[(df1.camera != 'lc1')& (df1.camera != 'lc2')]
df2 = df2[(df2.camera != 'lc1')& (df2.camera != 'lc2')]
if use_four: 
    df3 = df3[(df3.camera != 'lc1')& (df3.camera != 'lc2')]
    df4 = df4[(df2.camera != 'lc1')& (df4.camera != 'lc2')]

merge_cols = ['camera','target', 'annotated_car_id']
dims = ["OH","OL","OW"]
def compare_dims(row):
    for c in dims:
        row[f"DIFF_{c}"] = row[f"PRED_{c}"]-row[f"LD_{c}"]
    return row

df1 = df1.apply(lambda row: compare_dims(row), axis = 1)
df2 = df2.apply(lambda row: compare_dims(row), axis = 1)
for c in dims:
    df1[f"ABS_DIFF_{c}"] = df1[f"DIFF_{c}"].apply(abs)
for c in dims:
    df2[f"ABS_DIFF_{c}"] = df2[f"DIFF_{c}"].apply(abs)
    
if use_four:
    df3 = df3.apply(lambda row: compare_dims(row), axis = 1)
    df4 = df4.apply(lambda row: compare_dims(row), axis = 1)
    for c in dims:
        df3[f"ABS_DIFF_{c}"] = df3[f"DIFF_{c}"].apply(abs)
    for c in dims:
        df4[f"ABS_DIFF_{c}"] = df4[f"DIFF_{c}"].apply(abs)

df1['dist_base_bbox_diff_abs'] = df1['dist_base_bbox_diff'].apply(abs)
df2['dist_base_bbox_diff_abs'] = df2['dist_base_bbox_diff'].apply(abs)
df1['dist_nearest_corner_diff_abs'] = df1['dist_nearest_corner_diff'].apply(abs)
df2['dist_nearest_corner_diff_abs'] = df2['dist_nearest_corner_diff'].apply(abs)
if use_four:
    df3['dist_base_bbox_diff_abs'] = df3['dist_base_bbox_diff'].apply(abs)
    df4['dist_base_bbox_diff_abs'] = df4['dist_base_bbox_diff'].apply(abs)
    df3['dist_nearest_corner_diff_abs'] = df3['dist_nearest_corner_diff'].apply(abs)
    df4['dist_nearest_corner_diff_abs'] = df4['dist_nearest_corner_diff'].apply(abs)
df1 = df1[df1.dist_base_gt_bbox <=150 ]
df2 = df2[df2.dist_base_gt_bbox <=150 ]
if use_four:
    df3 = df3[df3.dist_base_gt_bbox <=150 ]
    df4 = df4[df4.dist_base_gt_bbox <=150 ]

df1_before_merge = len(df1)
df2_before_merge = len(df2)
print(df1_before_merge, df2_before_merge)

assert len(df1[df1.duplicated(merge_cols, keep=False)]) == 0, 'NO duplicate rows'
assert len(df2[df2.duplicated(merge_cols, keep=False)]) == 0, 'NO duplicate rows'
if use_four:
    assert len(df3[df3.duplicated(merge_cols, keep=False)]) == 0, 'NO duplicate rows'
    assert len(df4[df4.duplicated(merge_cols, keep=False)]) == 0, 'NO duplicate rows'

if use_four:
    mdf = df1.merge(df2, on=merge_cols).merge(df3, on=merge_cols).merge(df4, on=merge_cols)
else:
    mdf = df1.merge(df2, on=merge_cols)

df1 = df1[df1.apply(lambda row: (row['camera'], row['target'], row['annotated_car_id']) in zip(mdf['camera'],mdf['target'], mdf['annotated_car_id']), axis=1)]
df2 = df2[df2.apply(lambda row: (row['camera'], row['target'], row['annotated_car_id']) in zip(mdf['camera'],mdf['target'], mdf['annotated_car_id']), axis=1)]

if use_four:
    df3 = df3[df3.apply(lambda row: (row['camera'], row['target'], row['annotated_car_id']) in zip(mdf['camera'],mdf['target'], mdf['annotated_car_id']), axis=1)]
    df4 = df4[df4.apply(lambda row: (row['camera'], row['target'], row['annotated_car_id']) in zip(mdf['camera'],mdf['target'], mdf['annotated_car_id']), axis=1)]

assert len(df1) == len(df2)
if use_four:
    assert len(df3) == len(df4)
    assert len(df2) == len(df3)
df1 = df1.sort_values(by=merge_cols).reset_index()
df1.drop('index', axis=1, inplace=True)
df2 = df2.sort_values(by=merge_cols).reset_index()
df2.drop('index', axis=1, inplace=True)
if use_four:
    df3 = df3.sort_values(by=merge_cols).reset_index()
    df3.drop('index', axis=1, inplace=True)  
    df4 = df4.sort_values(by=merge_cols).reset_index()
    df4.drop('index', axis=1, inplace=True)  

823 823


### Processing track

In [22]:
all_cams = ['sc1','sc2','sc3','sc4']
car_track_lookup = dict()
for cam in all_cams:
    with open(f"hw7_saved_tracks/Seg23/{cam}/Seg23_{cam}_20231011.json","r") as f:
        tracks = json.load(f)
    car_to_track = dict()
    for k,v in tracks.items():
        for i in v:
            car_to_track[i] = k
    car_track_lookup[cam] = car_to_track

In [23]:
def get_car_track(row):
    lookup_str = f"{str(int(row.target)).zfill(4)}_{int(row.annotated_car_id-1)}" #Difference between python index 0 and matlab index 1
    if lookup_str in car_track_lookup[row.camera]:
        return car_track_lookup[row.camera][lookup_str]

In [45]:
lidar = True
use_what_lidar = ''
use_what_lidar = '_NON'

In [46]:
if lidar:
    columns_labels_dict = {
        f'STD_LD_OH{use_what_lidar}':'MSD Height (m)',  
        f'STD_LD_OL{use_what_lidar}':'MSD Length (m)', 
        f'STD_LD_OW{use_what_lidar}':'MSD Width (m)', 
        f'STD_LD_WB{use_what_lidar}':'MSD Wheelbase (m)', 
        f'STD/mean_LD_OH{use_what_lidar}':'MSD Height (%)',  
        f'STD/mean_LD_OL{use_what_lidar}':'MSD Length (%)', 
        f'STD/mean_LD_OW{use_what_lidar}':'MSD Width (%)', 
        f'STD/mean_LD_WB{use_what_lidar}':'MSD Wheelbase (%)'
    }
else:
     columns_labels_dict = {
        'STD_PRED_OH':'MSD Height (m)',  
        'STD_PRED_OL':'MSD Length (m)', 
        'STD_PRED_WWOM':'MSD Width (m)', 
        'STD_PRED_WB':'MSD Wheelbase (m)', 
        'STD/mean_PRED_OH':'MSD Height (%)',  
        'STD/mean_PRED_OL':'MSD Length (%)', 
        'STD/mean_PRED_WWOM':'MSD Width (%)', 
        'STD/mean_PRED_WB':'MSD Wheelbase (%)'
    }

In [47]:
def generate_out_df(df,target_df):
    out_df = pd.DataFrame(df).T
    renamed_cols = [columns_labels_dict[i] for i in out_df.columns]
    out_df.columns = renamed_cols
    out_df['Number of vehicles'] = int(len(target_df))
    with pd.option_context("max_colwidth", 1000):
        re = out_df.T.to_latex(index=True,
                      formatters={"name": str.upper},
                      float_format="{:.2f}".format,
                      multirow=True,
                      multicolumn=True,
                      multicolumn_format='c',
                      position='h',
                     bold_rows=True
        )
    return re

In [48]:
target_df = df1
# target_df = target_df[target_df.PRED_WB != -1] #Exclude the car without wheelbase
target_df['track'] = target_df.apply(lambda row: get_car_track(row), axis = 1)
target_df = target_df.dropna()
# target_df['PRED_WB'] = target_df['PRED_WB'].apply(abs)
if lidar:
    critical_dims = [f"LD_OH{use_what_lidar}",f"LD_OL{use_what_lidar}",f"LD_OW{use_what_lidar}"]
else:
    critical_dims = ["PRED_OH","PRED_OL","PRED_WWOM"]
all_tracks_std = []
all_tracks_percent_mean = []
for k in target_df.track.unique():
    if len(target_df[target_df.track == k]) < 2:
        continue #Avoid only one car in track
    print(target_df[target_df.track == k][critical_dims])
    std = np.std(target_df[target_df.track == k][critical_dims], axis = 0)
    avg = np.mean(target_df[target_df.track == k][critical_dims], axis = 0)
    percent = std/avg*100
    all_tracks_std.append(std.values)
    all_tracks_percent_mean.append(percent.values)

print(all_tracks_std)
print(all_tracks_percent_mean)
print('Number of tracks', len(all_tracks_std))
results = np.round(np.concatenate([np.mean(all_tracks_std,0),np.mean(all_tracks_percent_mean,0)]),2)
cols = [f"STD_{i}" for i in critical_dims] +  [f"STD/mean_{i}" for i in critical_dims]
agg_df_1 = pd.DataFrame.from_dict(dict(zip(cols, results)),orient='index')
print(generate_out_df(agg_df_1,target_df))

     LD_OH_NON  LD_OL_NON  LD_OW_NON
0         0.83       4.81       0.78
6         0.93       4.89       0.88
12        0.93       4.75       0.91
16        1.03       4.78       0.79
28        1.70       4.42       0.42
48        1.00       3.84       0.18
56        0.26       4.51       0.87
583       0.83       4.81       0.78
588       0.93       4.89       0.88
590       0.93       4.75       0.91
593       1.03       4.78       0.79
596       1.68       4.79       0.96
607       1.62       4.39       0.37
     LD_OH_NON  LD_OL_NON  LD_OW_NON
1         1.18       5.69       0.80
7         1.21       5.61       0.79
13        1.56       5.72       0.75
17        1.65       5.52       0.44
22        1.72       5.32       1.86
25        1.63       5.37       0.49
31        1.35       5.11       0.69
584       1.18       5.69       0.80
591       1.56       5.72       0.75
594       1.65       5.52       0.44
597       1.72       5.32       1.86
     LD_OH_NON  LD_OL_NON  LD_OW_NON
2

In [34]:
### OUR
Number of tracks 34
\begin{table}[h]
\begin{tabular}{lr}
\toprule
 & 0 \\
\midrule
\textbf{MSD Height (m)} & 0.09 \\
\textbf{MSD Length (m)} & 0.35 \\
\textbf{MSD Width (m)} & 0.08 \\
\textbf{MSD Height (%)} & 5.67 \\
\textbf{MSD Length (%)} & 6.96 \\
\textbf{MSD Width (%)} & 4.51 \\
\textbf{Number of vehicles} & 823.00 \\
\bottomrule
\end{tabular}
\end{table}


### LD\(_{prop}\)
Number of tracks 34
\begin{table}[h]
\begin{tabular}{lr}
\toprule
 & 0 \\
\midrule
\textbf{MSD Height (m)} & 0.00 \\
\textbf{MSD Length (m)} & 0.00 \\
\textbf{MSD Width (m)} & 0.00 \\
\textbf{MSD Height (%)} & 0.00 \\
\textbf{MSD Length (%)} & 0.00 \\
\textbf{MSD Width (%)} & 0.00 \\
\textbf{Number of vehicles} & 823.00 \\
\bottomrule
\end{tabular}
\end{table}

### LD
Number of tracks 34
\begin{table}[h]
\begin{tabular}{lr}
\toprule
 & 0 \\
\midrule
\textbf{MSD Height (m)} & 0.33 \\
\textbf{MSD Length (m)} & 1.17 \\
\textbf{MSD Width (m)} & 0.39 \\
\textbf{MSD Height (%)} & 39.36 \\
\textbf{MSD Length (%)} & 41.45 \\
\textbf{MSD Width (%)} & 54.14 \\
\textbf{Number of vehicles} & 823.00 \\
\bottomrule
\end{tabular}
\end{table}

SyntaxError: invalid syntax (3578496465.py, line 2)

In [2]:
path = Path('.')

In [3]:
track_path = Path(f'/mnt/c/Projects_raw_data/HW7/saved_tracks')

In [4]:
# intersection_begin = 3805
# with open(f"{str(track_path)}/{intersection_begin}_tracks_20230509.json","r") as f:
#     tracks = json.load(f)

In [5]:
#Load the results in here as a dataframe 
result_path = Path('compare_data')

In [7]:
#Name the columns
col_names = ["target", "annotated_car_id",
       "num_sym_pairs","bbox_2D_height",
             "reproj_error","gt_heading_angle",
             "pred_heading_angle","angle_difference",
             "dist_base_gt_bbox","dist_base_pred_bbox",
             "dist_base_bbox_diff","dist_nearest_corner_gt_bbox",
             "dist_nearest_corner_pred_bbox","dist_nearest_corner_diff",
             "iou","iou_bev","mounting_height","ds",
            "PRED_OL","PRED_OW","PRED_OH","PRED_WB","LD_OL","LD_OW","LD_OH"]

In [8]:
def get_car_track(row):
    lookup_str = f"{str(int(row.target)).zfill(4)}_{int(row.annotated_car_id-1)}" #Difference between python index 0 and matlab index 1
    if lookup_str in car_to_track:
        row['track'] = car_to_track[lookup_str]
    return row

In [9]:
def get_exist_in_both(intersection_begin,annotator,suffix):
    fname = f'{intersection_begin}_{annotator}_{suffix}.txt'
    df = pd.read_csv(result_path/fname, delim_whitespace=True, header=None)
    assert len(col_names) == df.shape[-1]
    df.columns = col_names
    gdf = df.dropna()
    gdf = gdf[gdf.PRED_WB != -1]
    return gdf[['target', 'annotated_car_id']]

In [10]:
def eval_consistency(annotator,merged_df):
    fname = f'{intersection_begin}_{annotator}_{suffix}.txt'
    df = pd.read_csv(result_path/fname, delim_whitespace=True, header=None)
    assert len(col_names) == df.shape[-1]
    df.columns = col_names
    df = df.apply(get_car_track, axis = 1)
    gdf = df.dropna()
    gdf = gdf[gdf.PRED_WB != -1]
    # align indices
    gdf = gdf.set_index(['target', 'annotated_car_id'])
    merged_df = merged_df.set_index(['target', 'annotated_car_id'])
    gdf = gdf[gdf.index.isin(merged_df.index)].reset_index()   
    print('Num records',len(gdf))
    
    critical_dims = ["PRED_OH","PRED_OL","PRED_OW","PRED_WB"]
    
    all_tracks_std = []
    all_tracks_percent_mean = []
    for i in gdf.track.unique():
        std = np.std(gdf[gdf.track == i][critical_dims], axis = 0)
        avg = np.mean(gdf[gdf.track == i][critical_dims], axis = 0)
        percent = std/avg*100
        all_tracks_std.append(std.values)
        all_tracks_percent_mean.append(percent.values)
#     print(np.mean(all_tracks_std,0),np.mean(all_tracks_percent_mean,0))
    return np.mean(all_tracks_std,0),np.mean(all_tracks_percent_mean,0)

In [11]:
annotators = ['thao','thao']
suffix = '20231013'
track_suffix = '20231011'
lidars = [True, False]
segment = "Seg23"
intersections = ["seg23_sc2"]
camera = "sc2"


for intersection_begin in intersections:
   #Processing track
    with open(f"{str(track_path)}/{segment}/{camera}/{segment}_{camera}_{track_suffix}.json","r") as f:
        tracks = json.load(f)
    car_to_track = dict()
    for k,v in tracks.items():
        for i in v:
            car_to_track[i] = k
            
    #Combined common results only for better results
    all_dfs = []    
    for i in annotators:
        one_df = get_exist_in_both(intersection_begin,i,suffix)
        all_dfs.append(one_df)
    merged_df = all_dfs[0].merge(all_dfs[1], on=['target', 'annotated_car_id'], how='inner')
    
    
    
    two_all_tracks_std = []
    two_all_tracks_percent_mean = []
    for i in annotators:
        std, percent_mean = eval_consistency(i,merged_df)
        two_all_tracks_std.append(std)
        two_all_tracks_percent_mean.append(percent_mean)
        
        
        
    two_all_tracks_std = np.array(two_all_tracks_std)
    two_all_tracks_percent_mean = np.array(two_all_tracks_percent_mean)
    print(intersection_begin, i, suffix)
    print(np.round(np.mean(two_all_tracks_std,0),2))
    print(np.around(np.mean(two_all_tracks_percent_mean,0),2))
    print("=================")
#     break


Num records 18
Num records 18
seg23_sc2 thao 20231013
[0.03 0.1  0.03 0.08]
[1.84 1.95 1.97 2.56]


In [ ]:
annotator = 'ceci'
suffix = 'before_corrections_OFFICIAL_20231002'

In [ ]:
fname = f'{intersection_begin}_{annotator}_{suffix}.txt'
print(fname)

In [ ]:
df = pd.read_csv(result_path/fname, delim_whitespace=True, header=None)

In [ ]:

assert len(col_names) == df.shape[-1]

In [ ]:
df.columns = col_names

In [ ]:
car_to_track = dict()
for k,v in tracks.items():
    for i in v:
        car_to_track[i] = k

In [ ]:
df = df.apply(get_car_track, axis = 1)

In [ ]:
gdf = df.dropna()

In [ ]:
### Drop case where wheelbase = -1

In [ ]:
gdf = gdf[gdf.PRED_WB != -1]

In [ ]:
critical_dims = ["PRED_OH","PRED_OL","PRED_OW","PRED_WB"]

In [ ]:
all_tracks_std = []
all_tracks_percent_mean = []
for i in gdf.track.unique():
    std = np.std(gdf[gdf.track == i][critical_dims], axis = 0)
    avg = np.mean(gdf[gdf.track == i][critical_dims], axis = 0)
    percent = std/avg*100
    all_tracks_std.append(std.values)
    all_tracks_percent_mean.append(percent.values)

In [ ]:
all_tracks_std = np.array(all_tracks_std)
all_tracks_percent_mean = np.array(all_tracks_percent_mean)

In [ ]:
all_tracks_std = pd.DataFrame(all_tracks_std)
all_tracks_std.columns = critical_dims

In [ ]:
all_tracks_percent_mean = pd.DataFrame(all_tracks_percent_mean)
all_tracks_percent_mean.columns = critical_dims

In [ ]:
all_tracks_std.describe()

In [ ]:
all_tracks_percent_mean.describe()